# CN-Only Model with Dedicated Optuna Optimization

This notebook trains a brain age prediction model **exclusively on healthy controls (CN)**,
with hyperparameters optimized specifically for CN data (not reused from the mixed pipeline).

**Motivation:** The previous CN-only experiment reused hyperparameters optimized for the full
1120-subject cohort (CN + AD + FTD). With only 473 CN subjects, those hyperparameters may not
be optimal. This experiment tests whether dedicated optimization changes the results.

**Evaluation strategy:**
- **CN:** 53 held-out test patients (other 473 CN used for training)
- **AD:** All 422 patients (none used for training)
- **FTD:** All 297 patients (none used for training)

**Steps:**
1. Load cohort & main splits, filter CN trainval
2. Optuna VAE optimization on CN data
3. K-Fold VAE training + final VAE on CN
4. Optuna XGBoost optimization on CN data
5. Train final XGBoost on CN
6. Evaluate on CN test + ALL AD + ALL FTD
7. Compare with reused-params results

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent  # Code/
REPO = ROOT.parent                  # Thesis-comp-sci/

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import Paths, ExperimentConfig
from src.utils_seed import set_global_seed
from src.cohort import build_final_cohort_df
from src.splits import make_kfold_splits
from src.data_io import load_fc_vectors_for_ids
from src.vae_optuna_age import run_vae_optuna_for_age, folds_ids_to_indices, _HIDDEN_DIMS_MAP
from src.vae_train import train_vae_kfold, train_vae_final
from src.embeddings import encode_mu, save_embeddings
from src.optuna_xgb import tune_xgb_with_cv
from src.xgb_train import build_feats, clean_xy, train_xgb
from src.metrics import regression_metrics

print("Imports OK")

2026-02-22 20:39:13.111932: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-22 20:39:13.652113: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-22 20:39:14.792725: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Imports OK


/home/nico/Desktop/5to/TesisFinal/.tesis-final-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)

cfg = ExperimentConfig(
    seed=42,
    diagnoses_to_use=("CN", "AD", "FTD"),
    test_size=0.10,
    k_folds=5,
    fisher_z=True,
    reuse_artifacts=True,
    use_optuna=True,
    optuna_vae_trials=50,
    optuna_xgb_trials=100,
)

set_global_seed(cfg.seed)

EXP_DIR = paths.out_dir / "experiments" / "exp2_cn_only_optuna"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {EXP_DIR}")

Output directory: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments/exp2_cn_only_optuna


## 1. Load cohort & splits

In [3]:
cohort_df = build_final_cohort_df(
    paths.excel_path,
    paths.fc_folder,
    paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)
print(f"Total cohort: N={len(cohort_df)}")
print(f"Diagnoses: {cohort_df['diagnosis'].value_counts().to_dict()}")

splits_file = paths.out_dir / "splits" / f"splits_seed{cfg.seed}_test{cfg.test_size}.json"
with open(splits_file) as f:
    splits_data = json.load(f)

trainval_ids = splits_data["holdout"]["trainval_ids"]
test_ids     = splits_data["holdout"]["test_ids"]
print(f"Main split -> trainval: {len(trainval_ids)}, test: {len(test_ids)}")

Total cohort: N=1245
Diagnoses: {'CN': 526, 'AD': 422, 'FTD': 297}
Main split -> trainval: 1120, test: 125


In [4]:
cn_all_ids      = set(cohort_df.loc[cohort_df["diagnosis"] == "CN", "record_id"])
cn_trainval_ids = sorted([rid for rid in trainval_ids if rid in cn_all_ids])
cn_test_ids     = sorted([rid for rid in test_ids     if rid in cn_all_ids])

ad_all_ids  = sorted(cohort_df.loc[cohort_df["diagnosis"] == "AD",  "record_id"].tolist())
ftd_all_ids = sorted(cohort_df.loc[cohort_df["diagnosis"] == "FTD", "record_id"].tolist())

eval_ids = cn_test_ids + ad_all_ids + ftd_all_ids

print(f"CN trainval (from main split): {len(cn_trainval_ids)}")
print(f"CN test     (from main split): {len(cn_test_ids)}")
print(f"AD  (all, for evaluation):     {len(ad_all_ids)}")
print(f"FTD (all, for evaluation):     {len(ftd_all_ids)}")
print(f"Total evaluation set:          {len(eval_ids)}")

# Verify no overlap between CN training and evaluation
assert len(set(cn_trainval_ids) & set(eval_ids)) == 0, "Data leakage detected!"
print("\nNo overlap between training and evaluation sets.")

CN trainval (from main split): 473
CN test     (from main split): 53
AD  (all, for evaluation):     422
FTD (all, for evaluation):     297
Total evaluation set:          772

No overlap between training and evaluation sets.


In [5]:
cn_kfold = make_kfold_splits(cn_trainval_ids, seed=cfg.seed, k=cfg.k_folds)
print(f"CN KFold: {len(cn_kfold)} folds")
for i, fold in enumerate(cn_kfold):
    print(f"  Fold {i}: train={len(fold['train_ids'])}, val={len(fold['val_ids'])}")

CN KFold: 5 folds
  Fold 0: train=378, val=95
  Fold 1: train=378, val=95
  Fold 2: train=378, val=95
  Fold 3: train=379, val=94
  Fold 4: train=379, val=94


## 2. Load FC & T1w data

In [6]:
print("Loading FC vectors ...")
all_record_ids = cohort_df["record_id"].tolist()
X_all = load_fc_vectors_for_ids(
    paths.fc_folder,
    all_record_ids,
    apply_fisher_z=cfg.fisher_z,
)
print(f"FC matrix: {X_all.shape}")

id_to_idx = {rec_id: i for i, rec_id in enumerate(all_record_ids)}

cn_trainval_idx = [id_to_idx[rid] for rid in cn_trainval_ids]
eval_idx        = [id_to_idx[rid] for rid in eval_ids]

X_cn_trainval = X_all[cn_trainval_idx]
X_eval        = X_all[eval_idx]
print(f"X_cn_trainval: {X_cn_trainval.shape}")
print(f"X_eval:        {X_eval.shape}")

Loading FC vectors ...
FC matrix: (1245, 6670)
X_cn_trainval: (473, 6670)
X_eval:        (772, 6670)


In [7]:
def get_data_for_ids(ids):
    df_sub = (cohort_df[cohort_df["record_id"].isin(ids)]
              .set_index("record_id").loc[ids].reset_index())
    y   = df_sub["age"].values
    sex = (df_sub["sex"] == "M").astype(float).values
    diag = df_sub["diagnosis"].map({"CN": 0, "AD": 1, "FTD": 2}).values
    T1   = df_sub[[c for c in df_sub.columns if c.startswith("t1_")]].values
    return y, sex, diag, T1

y_cn_tr, sex_cn_tr, _, T1_cn_tr = get_data_for_ids(cn_trainval_ids)
y_eval, _, diag_eval, T1_eval   = get_data_for_ids(eval_ids)

print(f"CN trainval: y shape={y_cn_tr.shape}, T1 shape={T1_cn_tr.shape}")
print(f"Eval set:    y shape={y_eval.shape}, T1 shape={T1_eval.shape}")
print(f"CN trainval age range: {y_cn_tr.min():.1f} - {y_cn_tr.max():.1f} (mean={y_cn_tr.mean():.1f})")

CN trainval: y shape=(473,), T1 shape=(473, 116)
Eval set:    y shape=(772,), T1 shape=(772, 116)
CN trainval age range: 18.0 - 92.0 (mean=63.5)


## 3. Optuna VAE optimization (on CN-only data)

This optimizes the VAE hyperparameters specifically for the 473 CN subjects,
using 5-fold CV with downstream age prediction MAE as the objective.

**This is the key difference from the previous experiment**, which reused
hyperparameters optimized for 1120 subjects across all diagnoses.

In [8]:
vae_fixed = dict(
    l2_reg=2.897389671945472e-07,
    activation="elu",
    norm_kind="layernorm",
    recon_kind="mae",
    clipnorm=1.0,
    batch_size=64,
    max_epochs=80,
    patience=10,
)

xgb_fixed = {
    "n_estimators": 800,
    "max_depth": 5,
    "learning_rate": 0.02,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 1e-4,
    "reg_lambda": 1.0,
    "min_child_weight": 1.0,
    "gamma": 1.0,
}

folds_idx = folds_ids_to_indices(cn_trainval_ids, cn_kfold)
print(f"Fold indices ready: {len(folds_idx)} folds")
for i, (tr, va) in enumerate(folds_idx):
    print(f"  Fold {i}: train={len(tr)}, val={len(va)}")

Fold indices ready: 5 folds
  Fold 0: train=378, val=95
  Fold 1: train=378, val=95
  Fold 2: train=378, val=95
  Fold 3: train=379, val=94
  Fold 4: train=379, val=94


In [9]:
optuna_vae_dir = EXP_DIR / "optuna_vae"
best_vae_json  = optuna_vae_dir / "vae_optuna_age_best.json"

if best_vae_json.exists():
    print("Loading existing VAE Optuna results ...")
    with open(best_vae_json) as f:
        best_vae_result = json.load(f)
else:
    print(f"Running VAE Optuna ({cfg.optuna_vae_trials} trials) on CN-only data ...")
    print("This will take a while (each trial trains a VAE for each fold).")
    best_vae_result = run_vae_optuna_for_age(
        X_trainval_fc=X_cn_trainval,
        y_trainval=y_cn_tr,
        T1_trainval=T1_cn_tr,
        sex_trainval=sex_cn_tr,
        diag_trainval=None,
        folds_idx=folds_idx,
        out_dir=optuna_vae_dir,
        seed=cfg.seed,
        n_trials=cfg.optuna_vae_trials,
        vae_fixed=vae_fixed,
        xgb_fixed_params=xgb_fixed,
        use_sex=False,
        use_diag=False,
        use_t1w=True,
        study_name="cn_only_vae",
        storage_path=optuna_vae_dir / "cn_only_vae.db",
    )

print(f"\nBest CN-only VAE CV MAE: {best_vae_result['best_value_mae']:.3f}")
print(f"Best params: {best_vae_result['best_params']}")

Running VAE Optuna (50 trials) on CN-only data ...
This will take a while (each trial trains a VAE for each fold).


[I 2026-02-22 20:39:49,413] A new study created in RDB with name: cn_only_vae
Best trial: 48. Best value: 7.03619: 100%|██████████| 50/50 [1:42:41<00:00, 123.23s/it]


Best CN-only VAE CV MAE: 7.036
Best params: {'latent_dim': 32, 'hidden_dims': '256', 'beta_target': 0.14269302732394623, 'lr': 0.0006620731800491555, 'drop_rate': 0.1172969192455565, 'warmup_ep': 13, 'embedding_kind': 'z'}


In [10]:
bp = best_vae_result["best_params"]

cn_vae_hparams = dict(
    hidden_dims=_HIDDEN_DIMS_MAP[str(bp["hidden_dims"])],
    latent_dim=int(bp["latent_dim"]),
    beta_target=float(bp["beta_target"]),
    warmup_ep=int(bp["warmup_ep"]),
    l2_reg=float(vae_fixed["l2_reg"]),
    lr=float(bp["lr"]),
    recon_kind=str(vae_fixed["recon_kind"]),
    drop_rate=float(bp["drop_rate"]),
    activation=str(vae_fixed["activation"]),
    norm_kind=str(vae_fixed["norm_kind"]),
    batch_size=int(vae_fixed["batch_size"]),
    clipnorm=float(vae_fixed["clipnorm"]),
)

print("CN-optimized VAE hyperparameters:")
for k, v in cn_vae_hparams.items():
    print(f"  {k}: {v}")

# Compare with main pipeline params
print("\n--- Comparison with main pipeline VAE ---")
main_hparams_path = paths.out_dir / "vae" / "vae_final_trainval_optuna" / "hparams.json"
with open(main_hparams_path) as f:
    main_hp = json.load(f)
for k in ["hidden_dims", "latent_dim", "beta_target", "lr", "drop_rate", "warmup_ep"]:
    cn_v = cn_vae_hparams.get(k, "N/A")
    main_v = main_hp.get(k, "N/A")
    marker = " <-- different" if str(cn_v) != str(main_v) else ""
    print(f"  {k:15s}: CN={cn_v}, Main={main_v}{marker}")

CN-optimized VAE hyperparameters:
  hidden_dims: [256]
  latent_dim: 32
  beta_target: 0.14269302732394623
  warmup_ep: 13
  l2_reg: 2.897389671945472e-07
  lr: 0.0006620731800491555
  recon_kind: mae
  drop_rate: 0.1172969192455565
  activation: elu
  norm_kind: layernorm
  batch_size: 64
  clipnorm: 1.0

--- Comparison with main pipeline VAE ---
  hidden_dims    : CN=[256], Main=[512] <-- different
  latent_dim     : CN=32, Main=64 <-- different
  beta_target    : CN=0.14269302732394623, Main=0.056663247229966504 <-- different
  lr             : CN=0.0006620731800491555, Main=0.001892443497356961 <-- different
  drop_rate      : CN=0.1172969192455565, Main=0.036861053246000725 <-- different
  warmup_ep      : CN=13, Main=73 <-- different


## 4. Train VAE: K-Fold + Final model on CN

In [11]:
cn_vae_kfold_dir = EXP_DIR / "vae_kfold"
kfold_summary_path = cn_vae_kfold_dir / "kfold_summary.json"

if kfold_summary_path.exists():
    print("Loading existing K-Fold summary ...")
    with open(kfold_summary_path) as f:
        kfold_summary = json.load(f)
else:
    print("Training VAE K-Fold on CN data ...")
    kfold_summary = train_vae_kfold(
        X_cn_trainval,
        out_dir=cn_vae_kfold_dir,
        seed=cfg.seed,
        k=cfg.k_folds,
        fold_indices=[(np.array(tr), np.array(va)) for tr, va in folds_idx],
        max_epochs=300,
        **cn_vae_hparams,
    )

suggested_epochs = kfold_summary["suggested_epochs_for_final"]
print(f"\nK-Fold summary:")
print(f"  Mean val recon loss: {kfold_summary['mean_best_val_recon_loss']:.4f}")
print(f"  Mean val Pearson:    {kfold_summary['mean_best_val_pearson']:.4f}")
print(f"  Suggested epochs for final training: {suggested_epochs}")

Training VAE K-Fold on CN data ...

--- VAE Fold 1/5 ---
Epoch 1/300
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - kl_loss: 84.0429 - loss: 2281.0874 - recon_loss: 2281.0874
Epoch 1: Pearson=0.4925 (best=0.4925), Cosine=0.7109 (best=0.7109)
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - kl_loss: 124.5562 - loss: 1946.5880 - recon_loss: 1946.5880 - val_kl_loss: 153.1006 - val_loss: 1423.6595 - val_recon_loss: 1423.6595 - learning_rate: 6.6207e-04 - val_pearson: 0.4925 - val_cosine: 0.7109
Epoch 2/300
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - kl_loss: 159.9434 - loss: 1506.5416 - recon_loss: 1504.7859
Epoch 2: Pearson=0.5830 (best=0.5830), Cosine=0.7545 (best=0.7545)
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - kl_loss: 163.8291 - loss: 1475.3978 - recon_loss: 1473.5995 - val_kl_loss: 161.1931 - val_loss: 1280.6826 - val_recon_loss: 1278.9132 - learning_rate: 6.6207e-04 - val_pearson: 0.5830 - val_cosine: 0.7545
Epoch 3/300
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - kl_loss: 161.6000 - loss: 1373.3895 - recon_

In [12]:
cn_vae_final_dir = EXP_DIR / "vae_final"
weights_path = cn_vae_final_dir / "vae.weights.h5"

if weights_path.exists():
    print("Loading existing final VAE ...")
    from src.vae_train import load_vae_from_dir
    cn_vae = load_vae_from_dir(cn_vae_final_dir)
else:
    print(f"Training final VAE on all {len(cn_trainval_ids)} CN subjects ({suggested_epochs} epochs) ...")
    cn_vae, cn_vae_history = train_vae_final(
        X_cn_trainval,
        out_dir=cn_vae_final_dir,
        epochs=suggested_epochs,
        seed=cfg.seed,
        **cn_vae_hparams,
    )
    print(f"Final VAE training complete. Last loss: {cn_vae_history['loss'][-1]:.4f}")

print("VAE ready.")

Training final VAE on all 473 CN subjects (117 epochs) ...
Epoch 1/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - kl_loss: 135.4028 - loss: 1821.1409 - recon_loss: 1821.1409 - learning_rate: 6.6207e-04
Epoch 2/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - kl_loss: 169.8144 - loss: 1413.5062 - recon_loss: 1411.6421 - learning_rate: 6.6207e-04
Epoch 3/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - kl_loss: 181.2152 - loss: 1306.4303 - recon_loss: 1302.4521 - learning_rate: 6.6207e-04
Epoch 4/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - kl_loss: 205.8483 - loss: 1260.0876 - recon_loss: 1253.3093 - learning_rate: 6.6207e-04
Epoch 5/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - kl_loss: 214.6586 - loss: 1237.7086 - recon_loss: 1228.2839 - learning_rate: 6.6207e-04
Epoch 6/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - kl_loss: 204.2564 - loss: 1215.8092 - recon_loss: 1204.5991 - learning_rate: 6.6207e-04
Epoch 7/117
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - kl_loss: 195.8906 - loss: 1189.5439 - recon_loss:

## 5. Extract embeddings

In [13]:
Z_cn_tr = encode_mu(cn_vae.encoder, X_cn_trainval)
Z_eval  = encode_mu(cn_vae.encoder, X_eval)
print(f"CN trainval embeddings: {Z_cn_tr.shape}")
print(f"Eval embeddings:        {Z_eval.shape}")

emb_dir = EXP_DIR / "embeddings"
emb_dir.mkdir(parents=True, exist_ok=True)
save_embeddings(emb_dir / "Z_cn_trainval", cn_trainval_ids, Z_cn_tr)
save_embeddings(emb_dir / "Z_eval", eval_ids, Z_eval)

CN trainval embeddings: (473, 32)
Eval embeddings:        (772, 32)


## 6. Optuna XGBoost optimization (on CN-only embeddings + T1w)

In [14]:
X_cn_xgb = build_feats(Z=Z_cn_tr, T1=T1_cn_tr)
X_cn_xgb_clean, y_cn_tr_clean = clean_xy(X_cn_xgb, y_cn_tr)
print(f"XGB train features: {X_cn_xgb_clean.shape}")
print(f"XGB train labels:   {y_cn_tr_clean.shape}")

XGB train features: (473, 148)
XGB train labels:   (473,)


In [15]:
optuna_xgb_dir = EXP_DIR / "optuna_xgb"
optuna_xgb_dir.mkdir(parents=True, exist_ok=True)
xgb_best_path = optuna_xgb_dir / "xgb_best_cn.json"

if xgb_best_path.exists():
    print("Loading existing XGBoost Optuna results ...")
    with open(xgb_best_path) as f:
        cn_xgb_best = json.load(f)["best_params"]
else:
    print(f"Running XGBoost Optuna ({cfg.optuna_xgb_trials} trials) on CN-only data ...")
    cn_xgb_best = tune_xgb_with_cv(
        X_cn_xgb_clean, y_cn_tr_clean,
        seed=cfg.seed,
        n_trials=cfg.optuna_xgb_trials,
        k_folds=cfg.k_folds,
        out_path=xgb_best_path,
    )

print(f"\nBest CN-only XGBoost params:")
for k, v in cn_xgb_best.items():
    print(f"  {k}: {v}")

# Compare with main pipeline
print("\n--- Comparison with main pipeline XGBoost ---")
with open(paths.out_dir / "optuna" / "xgb_best_cv.json") as f:
    main_xgb = json.load(f)["best_params"]
for k in ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"]:
    cn_v = cn_xgb_best.get(k, "N/A")
    main_v = main_xgb.get(k, "N/A")
    print(f"  {k:18s}: CN={cn_v}, Main={main_v}")

Running XGBoost Optuna (100 trials) on CN-only data ...


Best trial: 81. Best value: 6.97471: 100%|██████████| 100/100 [36:37<00:00, 21.97s/it] 


Best CN-only XGBoost params:
  n_estimators: 2414
  max_depth: 6
  learning_rate: 0.013354559691529854
  subsample: 0.5000367063365547
  colsample_bytree: 0.9194360628822144
  reg_alpha: 0.2511581830736604
  reg_lambda: 0.5306237937590532
  min_child_weight: 0.8371346367120922
  gamma: 0.4821522587683412
  tree_method: hist
  random_state: 42
  eval_metric: mae

--- Comparison with main pipeline XGBoost ---
  n_estimators      : CN=2414, Main=2103
  max_depth         : CN=6, Main=6
  learning_rate     : CN=0.013354559691529854, Main=0.011042092255313062
  subsample         : CN=0.5000367063365547, Main=0.5021328242984182
  colsample_bytree  : CN=0.9194360628822144, Main=0.9505971249574193


## 7. Train final XGBoost & evaluate

In [16]:
model_cn = train_xgb(X_cn_xgb_clean, y_cn_tr_clean, cn_xgb_best)
print(f"XGBoost trained on {len(y_cn_tr_clean)} CN subjects with CN-optimized params")

XGBoost trained on 473 CN subjects with CN-optimized params


In [17]:
eval_df = (cohort_df[cohort_df["record_id"].isin(eval_ids)]
           .set_index("record_id").loc[eval_ids].reset_index())

X_eval_xgb = build_feats(Z=Z_eval, T1=T1_eval)
X_eval_clean, y_eval_clean = clean_xy(X_eval_xgb, y_eval)

y_pred_eval = model_cn.predict(X_eval_clean)

eval_df["y_true"]        = y_eval_clean
eval_df["y_pred"]        = y_pred_eval
eval_df["brain_age_gap"] = eval_df["y_pred"] - eval_df["y_true"]

print("=" * 70)
print("CN-ONLY MODEL WITH DEDICATED OPTUNA — RESULTS")
print("=" * 70)

exp2_optuna_results = {}
for diag in ["CN", "AD", "FTD"]:
    sub_df = eval_df[eval_df["diagnosis"] == diag]
    if len(sub_df) == 0:
        continue
    m = regression_metrics(sub_df["y_true"].values, sub_df["y_pred"].values)
    gap_mean = sub_df["brain_age_gap"].mean()
    gap_std  = sub_df["brain_age_gap"].std()

    print(f"\n{diag} (N={len(sub_df)}):")
    print(f"  MAE:  {m['MAE']:.2f} years")
    print(f"  RMSE: {m['RMSE']:.2f} years")
    print(f"  R²:   {m['R2']:.3f}")
    print(f"  Pearson: {m['Pearson']:.3f}")
    print(f"  Brain Age Gap: {gap_mean:+.2f} ± {gap_std:.2f} years")

    exp2_optuna_results[diag] = {
        "n": len(sub_df),
        "mae": float(m["MAE"]),
        "rmse": float(m["RMSE"]),
        "r2": float(m["R2"]),
        "pearson": float(m["Pearson"]),
        "brain_age_gap_mean": float(gap_mean),
        "brain_age_gap_std":  float(gap_std),
    }

print("\n" + "=" * 70)

results_path = EXP_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(exp2_optuna_results, f, indent=2)
print(f"Results saved: {results_path}")

pred_csv = EXP_DIR / "predictions.csv"
eval_df[["record_id", "diagnosis", "age", "y_true", "y_pred", "brain_age_gap"]].to_csv(
    pred_csv, index=False
)
print(f"Predictions saved: {pred_csv}")

CN-ONLY MODEL WITH DEDICATED OPTUNA — RESULTS

CN (N=53):
  MAE:  6.29 years
  RMSE: 7.44 years
  R²:   0.490
  Pearson: 0.715
  Brain Age Gap: -0.49 ± 7.49 years

AD (N=422):
  MAE:  7.32 years
  RMSE: 9.09 years
  R²:   -0.145
  Pearson: 0.181
  Brain Age Gap: -2.21 ± 8.83 years

FTD (N=297):
  MAE:  6.63 years
  RMSE: 8.62 years
  R²:   -0.081
  Pearson: 0.205
  Brain Age Gap: +1.71 ± 8.46 years

Results saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments/exp2_cn_only_optuna/results.json
Predictions saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments/exp2_cn_only_optuna/predictions.csv


## 8. Comparison: CN-Optuna vs Reused params vs Main pipeline

In [18]:
reused_path = paths.out_dir / "experiments" / "exp2_cn_only_results.json"
if reused_path.exists():
    with open(reused_path) as f:
        reused_results = json.load(f)
else:
    print("Warning: Reused-params results not found. Run rerun_exp2_cn_only.py first.")
    reused_results = {}

print("\n" + "=" * 80)
print("COMPARISON TABLE")
print("=" * 80)
print(f"{'Group':<6} {'Metric':<8} {'CN-Optuna':>12} {'Reused params':>14} {'Main (mixed)':>14}")
print("-" * 60)

main_test_metrics_path = paths.out_dir / "metrics" / "test_metrics.json"
main_metrics = {}
if main_test_metrics_path.exists():
    with open(main_test_metrics_path) as f:
        main_metrics = json.load(f)

for diag in ["CN", "AD", "FTD"]:
    optuna_r = exp2_optuna_results.get(diag, {})
    reused_r = reused_results.get(diag, {})
    for metric in ["mae", "r2", "brain_age_gap_mean"]:
        opt_v = optuna_r.get(metric)
        reu_v = reused_r.get(metric)
        label = metric.upper().replace("_MEAN", "").replace("BRAIN_AGE_GAP", "GAP")

        opt_s = f"{opt_v:.2f}" if opt_v is not None else "—"
        reu_s = f"{reu_v:.2f}" if reu_v is not None else "—"

        # Main pipeline only has overall metrics, not per-group
        main_s = "—"
        if diag == "CN" and metric == "mae" and main_metrics:
            main_s = f"{main_metrics.get('MAE', '—')}"

        print(f"{diag:<6} {label:<8} {opt_s:>12} {reu_s:>14} {main_s:>14}")
    print()


COMPARISON TABLE
Group  Metric      CN-Optuna  Reused params   Main (mixed)
------------------------------------------------------------
CN     MAE              6.29           6.03              —
CN     R2               0.49           0.50              —
CN     GAP             -0.49          -0.42              —

AD     MAE              7.32           7.36              —
AD     R2              -0.15          -0.17              —
AD     GAP             -2.21          -1.86              —

FTD    MAE              6.63           6.93              —
FTD    R2              -0.08          -0.15              —
FTD    GAP              1.71           1.84              —



## 9. Summary & interpretation

After running this notebook, compare:

1. **CN-Optuna** (this notebook): VAE + XGBoost hyperparameters optimized specifically for 473 CN subjects
2. **Reused params** (previous experiment): Hyperparameters from the main pipeline (1120 subjects, all diagnoses)
3. **Main pipeline**: Mixed training on 1120 subjects (CN + AD + FTD)

Key questions:
- Does dedicated optimization improve the CN-only model?
- Does the brain age gap now show the expected direction (positive for AD and FTD)?
- Is the R² still negative for clinical groups?